# Herstelanalyse, Project R.E.M.

**Doel**: Herstelt het lichaam sneller van stress wanneer je naar muziek luistert?

**Stadia**:
1. Elke minuut een activiteitslabel geven (Slaap / Rust / Licht / Matig / Zwaar)
2. Persoonlijke herstelcurve bepalen: hoe snel herstelt deze persoon normaal?
3. Vergelijken: was het herstel tijdens een muziekluistersessie sneller dan normaal?

Voer eerst `python scripts/analysis/pipeline.py` uit om de CSV-bestanden aan te maken.

---

### Wat meten we?

Na stress of inspanning keert het stresssignaal terug naar een **persoonlijk rustniveau** — de stresswaarde waar jij normaal naartoe zakt. Die vloerwaarde noemen we de **asymptoot**.

Twee begrippen om goed uit elkaar te houden:

| Begrip | Wat het is |
|--------|-----------|
| **Asymptoot** (rustniveau) | De stresswaarde waar jij normaal naartoe terugkeert — jouw persoonlijke vloer. Een getal, bv. 30. |
| **Baseline herstelcurve** | Hoe snel jij op een dag *zonder muziek* naar die vloer terugzakt — uitgedrukt als **tau**. Een snelheid, geen waarde. |

**De baseline in dit notebook is altijd de verwachte herstelsnelheid zonder muziek.** We meten of muziek luisteren dat proces versnelt.

Het centrale meetgetal is de **tau-voordeel**:
- **Positief** = sneller herstel met muziek
- **Rond nul** = geen effect
- **Negatief** = trager herstel, of de meting mislukte

### Wat is tau?

Tau is de **tijdconstante van de herstelcurve**. Kleiner betekent een sneller herstel. Een tau van 10 betekent snel herstel, een tau van 100 is traag.

$tauvoordeel = tauverwacht - tauwerkelijk$

### *r²* als kwaliteitsscore

Elke sessie heeft een **r2_actual**: hoe goed de herstelcurve gefit kon worden. Als **r2_actual bijna nul** is, is de **tau_actual** onbetrouwbaar. Gebruik enkel sessies met r2 > 0.05 voor conclusies.

### Woordenlijst — kernbegrippen

Alle termen die door het hele notebook gebruikt worden, op één plek:

| Begrip | Definitie | Concreet voorbeeld |
|--------|-----------|-------------------|
| **Asymptoot** | Jouw persoonlijke stressvloer: de waarde waar het stresssignaal normaal naartoe terugkeert. Een getal, geen snelheid. | bosbes: ~30 stress-eenheden |
| **Tau (τ)** | De tijdconstante van de herstelcurve: hoeveel minuten het duurt om het grootste deel van de daling te maken. *Kleiner = sneller.* | τ = 10 min = snel · τ = 100 min = traag |
| **Tau_verwacht** | Tau gemeten op niet-sessiedagen — jouw normale herstelsnelheid *zonder* muziek. De referentie. | bosbes na Slaap: τ ≈ 98 min |
| **Tau_werkelijk** | Tau gemeten *tijdens* een muziekluistersessie. | bosbes, 29 jan, Energy: τ ≈ 3 min |
| **Voordeel (advantage)** | tau_verwacht − tau_werkelijk. Positief = muziek versnelde het herstel. | +95 min = muziek was ~95 min sneller dan normaal |
| **r²** | Hoe goed de exponentiële curve past op de data. 0 = past helemaal niet, 1 = perfect. Onder 0.05 → tau is onbetrouwbaar. | r² = 0.82 = gebruik dit · r² = 0.01 = negeer dit |
| **Pre_state** | De activiteitstoestand in de minuten vóór de sessie startte. Bepaalt welke baseline-curve gebruikt wordt. | Slaap = vroege ochtend, laag activiteitsniveau |
| **Pre_stress_mean** | De gemiddelde stresswaarde in de 60 minuten vóór de sessie. Geeft aan hoe gestresseerd je was bij het begin. | 58 = hoog · 19 = al bijna rustig |
| **Baseline herstelcurve** | De verwachte herstelcurve op niet-sessiedagen — de referentielijn waartegen sessies vergeleken worden. | Gestippelde witte lijn in de grafieken |
| **Reliable** | Sessies die beide kwaliteitsfilters doorstaan: r² > 0.05 én pre_stress ≥ asymptoot. Geeft het eerlijkste gemiddelde. | Van 19 sessies zijn er typisch 8 reliable |

### Voor het uitvoeren van dit notebook

Voer de pijplijn uit voor de deelnemers die je wilt analyseren:

```bash
python3 scripts/analysis/pipeline.py --participants bosbes kokosnoot --skip-extraction
```

De cel hieronder maakt de CSV-bestanden aan in `data/analysis/{codenaam}/`.

---

## Hoe werkt dit notebook? — Stap voor stap

| Stap | Wat er gebeurt | Waarom |
|------|---------------|--------|
| **1. Laden & filteren** | Data inladen, onbetrouwbare sessies wegfilteren (r² ≤ 0.05 of pre_stress < asymptoot) | Ruis verwijderen zodat het gemiddelde geen artefacten bevat |
| **2. Activiteitstoestand per dag** | Gestapelde staafgrafiek: hoeveel tijd in elke toestand | Controleren of de tijdsverdeling logisch eruitziet — dit is geen bewijs, enkel een sanity check |
| **3. Persoonlijke herstelcurves** | Exponentiële curve fitten op herstelmomenten op niet-sessiedagen | De *baseline* bepalen: hoe snel herstelt deze persoon normaal, zonder muziek? |
| **4. Baseline vs sessies** | Gestippelde witte lijn (normaal) vs gekleurde lijnen (met muziek) | Visueel zien of sessiedalingen sneller gaan dan verwacht |
| **5. Samenvatting voordelen** | Histogram + boxplots per afspeellijsttype en per préstaat | Is het effect consistent? Werkt een bepaald type beter? |
| **6. Statistische toetsen** | t-toets, ANOVA, gemengd-effectenmodel | Is het effect groter dan wat je door toeval zou verwachten? |
| **7. Watervalgrafiek** | Per sessie: verwachte τ (donker) vs werkelijke τ (gekleurd, faded = onbetrouwbaar) | Elk resultaat in één oogopslag |
| **8. Stemming vs herstel** | Scatterplot: fysiologisch voordeel vs hoe de deelnemer zich voelde | Gaan sneller herstel en beter gevoel samen? |

> De filters uit stap 1 (`reliable`) gelden voor alle stappen erna. Het scorebord toont meteen hoeveel sessies de filter overleven.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats

# Add analysis module to path
sys.path.insert(0, str(Path().resolve().parent / 'scripts' / 'analysis'))
sys.path.insert(0, str(Path().resolve().parent / 'scripts' / 'wearables'))

from activity_classifier import ActivityClassifier, STATES
from baselines import PersonBaseline
from session_effect import analyze_sessions, run_statistics

DATA_ROOT     = Path().resolve().parent / 'data'
ANALYSIS_ROOT = DATA_ROOT / 'analysis'

# Okabe-Ito kleurenblindvriendelijk palet
PARTICIPANTS = {
    'bosbes':     {'color': '#56B4E9', 'emoji': '🫐'},
    'kokosnoot':  {'color': '#E69F00', 'emoji': '🥥'},
    'limoen':     {'color': '#009E73', 'emoji': '🍋'},
    'peer':       {'color': '#CC79A7', 'emoji': '🍐'},
    'citroen':    {'color': '#F0E442', 'emoji': '🍋'},
    'kiwi':       {'color': '#D55E00', 'emoji': '🥝'},
    'watermeloen':{'color': '#0072B2', 'emoji': '🍉'},
    'aardbei':    {'color': '#56B4E9', 'emoji': '🍓'},
}

STATE_COLORS = {
    'Sleep': '#0072B2', 'Rest': '#56B4E9', 'Light': '#009E73',
    'Medium': '#E69F00', 'Heavy': '#D55E00'
}

# Dark theme matching project style
plt.rcParams.update({
    'figure.facecolor': '#0f1218', 'axes.facecolor': '#181e2a',
    'axes.edgecolor': '#232b3a', 'axes.labelcolor': '#c9d1d9',
    'axes.grid': True, 'grid.color': '#232b3a', 'grid.linewidth': 0.5,
    'text.color': '#c9d1d9', 'xtick.color': '#586475', 'ytick.color': '#586475',
    'legend.facecolor': '#181e2a', 'legend.edgecolor': '#232b3a',
    'font.family': 'monospace', 'font.size': 9, 'figure.dpi': 120,
})

---

## Pijplijnuitvoer inladen

Per deelnemer laden we drie bestanden:

| Bestand | Wat erin staat | In gewoon Vlaams |
|---------|----------------|-----------------|
| `classified_minutes.csv` | Elk minuut een activiteitsstate: Slaap, Rust, Licht, Matig, Zwaar of Onbekend | "Om 08:03 zat je rustig, om 09:14 was je licht actief" |
| `recovery_baselines.csv` | Herstelcurve per activiteitsstate: **tau**, asymptoot, t90, n_obs, r² | "Na het slapen duurt het bij bosbes normaal ~98 minuten voordat de stress wegzakt" |
| `session_effects.csv` | Per sessie: **tau-verwacht**, **tau-werkelijk**, **voordeel**, stemmingsdelta | "Op 29 jan met Energy-playlist: normaal 98 min, nu 3 min → voordeel +95 min" |

Deelnemers zonder data zijn nog niet door de pijplijn gelopen.

In [ ]:
---

## Stadium 1: Activiteitstoestanden

Elke minuut van de smartwatchdata krijgt een label:

| State | Kleur | Wanneer |
|-------|-------|---------| 
| **Slaap** | Diepblauw | 22:00 tot 08:00, hartslag ≤ 95 én geen stappen |
| **Rust** | Hemelblauw | Wakker maar stilzittend |
| **Licht** | Blauwgroen | Wandelen, lichte taken |
| **Matig** | Oranje | Stevig wandelen, fietsen |
| **Zwaar** | Vermiljoen | Intensief sporten |

Minuten zonder horloge (geen hartfrequentie en geen stress) zijn `Onbekend` en worden nergens meegeteld.

> **"Slaap" ≠ bewezen slaap.** De classifier gebruikt enkel tijdstip en beweging: 22:00–08:00 + geen stappen + hartslag ≤ 95 bpm = Slaap. Wie om 06:30 wakker ligt zonder te bewegen krijgt dus ook het Slaap-label. Het is een proxy voor "vroege ochtend, laag activiteitsniveau" — geen bevestigd slaapstadium. Voor deelnemers die het horloge 's nachts dragen is dit redelijk; voor deelnemers die het overdag dragen kan de vroege ochtend onterecht als Slaap worden gelabeld.

### De grafiek lezen

Elke kolom is één dag. De kleurblokken tonen welk deel van die dag in elke state was. Veel diepblauw = rustdag. Oranje/vermiljoen = trainingsdagen.

### Hoe werkt de classifier?

De classifier gebruikt **vaste drempelwaarden**:

| Label | Drempelwaarde |
|-------|--------------|
| Zwaar | Hartslag > 130 bpm, of stress > 70 én battery daalt snel |
| Matig | Hartslag > 100 bpm, of stress > 50 |
| Licht | Hartslag > 78 bpm, of > 5 stappen per minuut |
| Rust | Standaard als geen van bovenstaande geldt |
| Slaap | Tijdvenster 22:00–08:00 én niet duidelijk wakker (zie boven) |

Die getallen zijn handmatig ingesteld op een 5-minuten rollend mediaan, niet geleerd uit data.

**Het probleem**: drempels zijn niet per persoon gekalibreerd. Een atleet haalt HR 130 pas bij zwaar sporten; iemand minder getraind zit daar al bij matig wandelen. De labels zijn daardoor voor sommige deelnemers minder nauwkeurig, wat doorwerkt in de `pre_state`-classificatie en uiteindelijk in het berekende **tau-voordeel**.

### TODO: Random Forest op Garmin FIT-labels

Garmins FIT-bestanden bevatten een eigen `intensity`-label per minuut (sedentary / active / highly_active), berekend door het horloge op basis van het persoonlijke profiel. Dat zijn **gelabelde data per deelnemer** — grondwaarheid zonder handmatige drempels.

**Plan**:
1. Extraheer de FIT-intensiteitslabels via `fit_extractor.py` → `garmin_minute_activity.csv`
2. Train een **Random Forest** op HR + stress + body_battery als features, met de Garmin-labels als target
3. Swap het model in via de bestaande sklearn-interface van `ActivityClassifier` — geen andere code hoeft te veranderen

**Waarom het uitmaakt**: betere classifier → nauwkeurigere `pre_state` → betrouwbaardere `tau_expected` opzoeking → sterker herstelvoordeel-signaal.

> Verdieping: Troiano et al. (2008) *Physical activity in the United States measured by accelerometer*, Medicine & Science in Sports & Exercise.

In [ ]:

# ── Kwaliteitsfilters ─────────────────────────────────────────────────────────
# reliable = sessies die we écht kunnen interpreteren:
#   1. r² > 0.05  → de exponentiële fit was goed genoeg om tau te vertrouwen
#   2. pre_stress ≥ asymptoot → stress was verhoogd vóór de sessie; anders geeft
#      het model een artefact (de curve heeft nergens naartoe te dalen)

if not all_effects.empty:
    # Bouw asymptoot-opzoeking per (deelnemer, préstaat)
    asym_map = {}
    for name, data in all_data.items():
        bl = data.get('baselines')
        if bl is not None:
            for _, row in bl[bl['signal'] == 'stress'].iterrows():
                asym_map[(name, row['from_state'])] = float(row['asymptote'])

    if 'asymptote' not in all_effects.columns:
        all_effects['asymptote'] = all_effects.apply(
            lambda r: asym_map.get((r.get('participant'), r.get('pre_state')), np.nan),
            axis=1)

    pre_stress_ok = (
        all_effects['pre_stress_mean'].isna() |
        all_effects['asymptote'].isna() |
        (all_effects['pre_stress_mean'] >= all_effects['asymptote'])
    )
    n_pre_dropped = int((~pre_stress_ok).sum())

    if 'r2_actual' in all_effects.columns:
        r2_ok = all_effects['r2_actual'] > 0.05
    else:
        r2_ok = pd.Series(True, index=all_effects.index)
    n_r2_dropped = int((~r2_ok).sum())

    reliable = all_effects[pre_stress_ok & r2_ok].dropna(subset=['advantage']).copy()

    # ── Scorebord ────────────────────────────────────────────────────────────
    print(f"Alle sessies geladen:             {len(all_effects)}")
    print(f"  – pre_stress < asymptoot:       {n_pre_dropped} verwijderd (modelartefact)")
    print(f"  – r² ≤ 0.05 (slechte fit):      {n_r2_dropped} verwijderd (onbetrouwbare tau)")
    print(f"Betrouwbare sessies (reliable):   {len(reliable)}")

    if len(reliable) > 1:
        t_stat, p_val = stats.ttest_1samp(reliable['advantage'], 0)
        sig = "✓ statistisch significant" if p_val < 0.05 else "✗ niet significant"
        print()
        print("━━ KERNRESULTAAT (alleen betrouwbare sessies) ━━━━━━━━━━━━")
        print(f"  Gemiddeld voordeel : {reliable['advantage'].mean():+.1f} min")
        print(f"  Standaardafwijking : {reliable['advantage'].std():.1f} min")
        print(f"  t-toets (n={len(reliable):<2})    : t={t_stat:.3f},  p={p_val:.4f}  {sig}")
        print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
else:
    reliable = pd.DataFrame()
    print("Geen data geladen. Voer pipeline.py eerst uit.")


**Verwachte uitvoer**: bosbes (7 sessies), kokosnoot (9 sessies), limoen (1 sessie).

De gecombineerde tabel bundelt alle sessies van alle deelnemers voor de statistische toetsen verderop.

**Wat je ziet voor bosbes en kokosnoot:**

- **bosbes** draagt het horloge 's nachts. De Slaap-band (donkerblauw) is consistent zichtbaar.
- **kokosnoot** draagt het horloge overdag, maar er is toch een Slaap-band. Vroege ochtendminuten (06:00 tot 08:00) vallen nog binnen het nachtvenster van de classifier en krijgen het Slaap-label. De ~20% niet-gedragen minuten zijn als `Onbekend` uitgefilterd.

**Let op — "Slaap" betekent hier niet noodzakelijk echt slapen.** De classifier gebruikt twee criteria: tijdvenster (22:00–08:00) én lage beweging (geen stappen, hartslag ≤ 95). Wie om 06:30 wakker ligt zonder te bewegen krijgt dus het Slaap-label. Het is een proxy voor "vroege ochtend, laag activiteitsniveau", geen bevestigd slaapstadium.

**Pre_state-verdeling over alle sessies (19 totaal):** Slaap: 6 (31%) · Rust: 7 (37%) · Licht: 4 (21%) · Matig: 1 (5%). Voor **bosbes** specifiek starten 5 van de 8 sessies (62%) als Slaap. Voor **kokosnoot** is dat slechts 1 van de 9 sessies — de meeste kokosnoot-sessies krijgen pre_state = Rust of Licht.

Als de Slaap-baseline zwak is (lage **n_obs** of **r² = 0**), zijn de voordeel-scores voor die sessies minder betrouwbaar.

In [ ]:
participants_with_data = [p for p, d in all_data.items() if d.get('classified') is not None]

if not participants_with_data:
    print("No classified data found. Run pipeline.py first.")
else:
    fig, axes = plt.subplots(len(participants_with_data), 1,
                             figsize=(14, 3 * len(participants_with_data)),
                             squeeze=False)
    fig.suptitle('Activity State Distribution per Participant', fontsize=12, y=1.01)

    for i, codename in enumerate(participants_with_data):
        ax = axes[i][0]
        df = all_data[codename]['classified']
        if 'activity_state' not in df.columns:
            ax.set_visible(False)
            continue

        # Stacked bar per day
        df_daily = df.copy()
        df_daily['date'] = df_daily.index.date
        daily_counts = (df_daily.groupby(['date', 'activity_state'])
                        .size().unstack(fill_value=0))
        # Reorder states
        cols_present = [s for s in STATES if s in daily_counts.columns]
        daily_pct = daily_counts[cols_present].div(daily_counts[cols_present].sum(axis=1), axis=0) * 100

        bottom = np.zeros(len(daily_pct))
        for state in cols_present:
            ax.bar(range(len(daily_pct)), daily_pct[state], bottom=bottom,
                   color=STATE_COLORS.get(state, '#888'), label=state, width=1.0)
            bottom += daily_pct[state].values

        ax.set_ylabel('%', fontsize=8)
        ax.set_ylim(0, 100)
        ax.set_xticks([])
        ax.set_title(f"{PARTICIPANTS[codename]['emoji']} {codename}", fontsize=10, loc='left')
        if i == 0:
            ax.legend(loc='upper right', ncol=5, fontsize=8)

    plt.tight_layout()
    plt.savefig(ANALYSIS_ROOT / 'state_distribution.png', bbox_inches='tight', dpi=150)
    plt.show()

**Wat je ziet voor bosbes en kokosnoot:**

- **bosbes** draagt het horloge 's nachts. De Slaap-band (donkerblauw) is consistent zichtbaar.
- **kokosnoot** draagt het horloge overdag, maar er is toch een Slaap-band. Vroege ochtendminuten (06:00 tot 08:00) vallen nog binnen het nachtvenster van de classifier en krijgen het Slaap-label. De ~20% niet-gedragen minuten zijn als `Onbekend` uitgefilterd.

Bijna alle muziekluistersessies starten 's ochtends vroeg en krijgen **pre_state = Slaap**. Als de Slaap-baseline zwak is (lage **n_obs** of **r² = 0**), zijn de voordeel-scores ook minder betrouwbaar.

### Twee dingen die allebei 'baseline' heten

Voordat we de herstelcurves bekijken, is het belangrijk om twee begrippen uit elkaar te houden:

| Begrip | Wat het is | In de grafiek |
|--------|-----------|---------------|
| **Rustniveau** (`asymptoot`) | De stresswaarde waar het lichaam normaal naartoe terugkeert — jouw persoonlijke vloer | Horizontale stippellijn onderaan |
| **Herstellingssnelheid** (`tau`) | Hoe snel de stress *naar die vloer daalt* op een dag zonder muziek | De helling van de curve |

```
Stress na het ontwaken (geen muziek):

  60 ──●  ← je wordt wakker (beginwaarde)
  50    \
  40     \   ← stress daalt naar rustniveau
  30 ─────●──────────────────── ← rustniveau (asymptoot, ~30)
       0      30 min      90 min

  tau    = hoeveel minuten het duurt om het grootste deel van die daling te maken
  voordeel = tau_normaal − tau_muziek
               positief = sneller bij de vloer aangekomen mét muziek
```

> **Het voordeel meet niet hoe laag de stress werd — maar hoe snel het er was.**

---

## Stadium 2: Persoonlijke herstelcurves

### Hoe werkt het?

Op **niet-sessiedagen** volgt de pijplijn het stresssignaal 90 minuten nadat een activiteitsniveau daalt (bv. Zwaar naar Rust na een training). Op die data wordt een exponentiële curve gefit:

$$\text{stress}(t) = \text{asymptoot} + (\text{start} - \text{asymptoot}) \times e^{-t/\tau}$$

De mediaan van alle gefitte **tau**-waarden per activiteitsstate is de persoonlijke **baseline** voor die state.

### De tabel lezen

| Kolom | Betekenis |
|-------|-----------|
| `from_state` | Activiteitsstate *vóór* de overgang |
| `signal` | Welk signaal: stress, hartfrequentie of lichaamsaccu |
| `tau_min` | **Tijdconstante in minuten**: kleiner = sneller herstel. 500 = meting mislukt |
| `asymptote` | Persoonlijk rustniveau voor dit signaal |
| `t_90_min` | Minuten tot 90% herstel (= tau × 2.3) |
| `n_obs` | Aantal overgangen waarop de curve gebaseerd is |
| `r_squared` | Hoe goed de curve past (0 = slecht, 1 = perfect) |

> Verdieping: Berntson et al. (1997) *Heart rate variability: origins, methods, and interpretive caveats*, Psychophysiology.

In [ ]:
participants_with_baselines = [p for p, d in all_data.items() if d.get('baselines') is not None]

if participants_with_baselines:
    all_baselines = pd.concat(
        [all_data[p]['baselines'].assign(participant=p) for p in participants_with_baselines],
        ignore_index=True
    )
    print(all_baselines.to_string(index=False))

**Wat opvalt:**

**bosbes, Slaap naar stress: tau = 97.8 min**
Na het ontwaken duurt het bijna 100 minuten voordat bosbes' stress van nature terugzakt. Hoog plafond: elke sessie waarbij stress sneller daalt, toont een groot voordeel. Maar **r² = 0** voor alle baselines van bosbes. De curve past niet goed, de tau-waarden zijn schattingen.

**kokosnoot, Slaap naar stress: tau = 9.4 min, n_obs = 2**
Gebouwd op slechts 2 overgangen. Erg fragiel. Alle ochtendse sessies van kokosnoot gebruiken deze baseline.

**tau = 500 = meting mislukt**
De curvefitter bereikte zijn maximum. Behandel deze waarden als "onbekend".

**r² = 0 ≠ geen data**
bosbes heeft honderden observaties maar het herstel volgt geen zuivere exponentiële vorm. Echte fysiologische hersteldata is rommelig.

> **Kanttekening**: r² = 0 voor alle signalen bij bosbes suggereert dat het exponentiële model niet de juiste keuze is. LOWESS of een tweeledig model zou beter passen.

In [ ]:
---

## Persoonlijke baseline per activatieniveau

Dit is het concept uit de onderzoeksplanning: *"Baseline per persoon uitwerken met verschillende activatieniveaus, zodat je kan zien welk activatieniveau voor de sessie bezig was en of muziek invloed heeft op normaal herstel."*

De pijplijn doet dit in drie stappen:

**Stap 1** — elke minuut een activatieniveau geven (`classified_minutes.csv`)

**Stap 2** — op niet-sessiedagen de herstelcurve per activatieniveau bepalen (`recovery_baselines.csv`)

**Stap 3** — voor elke sessie vergelijken: verwachte hersteltijd vs. werkelijke hersteltijd

| Kolom | Betekenis |
|-------|-----------| 
| `pre_state` | Activatieniveau vóór de sessie |
| `tau_expected` | Verwachte hersteltijd uit de baseline |
| `tau_actual` | Werkelijke hersteltijd tijdens de sessie |
| `advantage` | tau_expected min tau_actual (positief = muziek hielp) |

### Hoe lees je de grafiek hieronder?

Elke grafiek heeft één paneel per préstaat. Binnen elk paneel:

```
Stress
  ↑
60│·  ·  ·  ← startpunt: pre_stress_mean (gemiddeld vóór de sessie)
  │  \  \ ← grijze lijnen = ruwe data van niet-sessiedagen
40│   \
  │    \  ← gekleurde lijn = wat er tijdens een muziekluistersessie gebeurde
  │     \
25│──────────────── ← asymptoot (jouw persoonlijke stressvloer)
  └─────────────→ tijd (0 tot 90 minuten)
     gestippeld wit = baseline (normale herstelsnelheid zonder muziek)
```

| Wat je ziet | Wat het betekent |
|-------------|-----------------|
| **Gestippelde witte lijn** | De verwachte herstelcurve op basis van niet-sessiedagen — hoe snel stress *normaal* naar de vloer daalt |
| **Gekleurde lijn (vol, hoge alpha)** | Werkelijke sessie met r² > 0.05 — betrouwbaar resultaat |
| **Gekleurde lijn (faded, lage alpha)** | Werkelijke sessie met r² ≤ 0.05 — de curve paste niet goed, tau is ruis |
| **Stippellijn onderaan (blauw)** | De asymptoot — de stressvloer van deze deelnemer |
| **Lijn *onder* de gestippelde witte lijn** | Muziek versnelde het herstel — voordeel positief |
| **Lijn *boven* de gestippelde witte lijn** | Herstel ging trager dan normaal — of de meting mislukte |
| **Alle gekleurde lijnen beginnen op hetzelfde punt** | Ze starten allemaal vanuit dezelfde gemiddelde pre_stress — zo zijn ze vergelijkbaar |

> **Let op**: als de startwaarde (pre_stress_mean) al dicht bij of onder de asymptoot ligt, is er weinig ruimte om te dalen en geeft de curve een artefact. Die sessies worden door het `reliable`-filter verwijderd.

**De grafiek lezen:**

Elk paneel = één signaal. Elke groep staven = één activiteitsstate. Elke gekleurde staaf = één deelnemer.

- **Hogere staaf = trager herstel** vanuit die state
- **Staaf bij 500 = meting mislukt**, behandel als onbekend
- **Lichaamsaccu-staven staan bijna altijd op 500** en zijn niet bruikbaar

bosbes heeft een Slaap-tau van ~98 min, kokosnoot ~9 min. Dat geeft bosbes veel meer ruimte om een voordeel te tonen bij ochtendse sessies.

In [ ]:
import warnings
from scipy.optimize import curve_fit, OptimizeWarning

def _fit_exp_simple(t, y, asymptote):
    """Fit y = asymptote + (y0 - asym) * exp(-t/tau), return (tau, r2) or None."""
    if len(y) < 10:
        return None
    y0 = float(y[0])
    tau_est = max(float(t[-1]) / 3, 1.0)
    def model(t_, tau):
        return asymptote + (y0 - asymptote) * np.exp(-t_ / tau)
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", OptimizeWarning)
            popt, _ = curve_fit(model, t, y, p0=[tau_est], bounds=(0.1, 500), maxfev=2000)
        tau = float(popt[0])
        y_pred = model(t, tau)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r2 = float(1 - ss_res / ss_tot) if ss_tot > 0 else 0.0
        return tau, max(r2, 0.0)
    except (RuntimeError, ValueError):
        return None


STATE_ORDER = {'Sleep': 0, 'Rest': 1, 'Light': 2, 'Medium': 3, 'Heavy': 4}
MAX_WINDOWS = 8

for codename in participants_with_baselines:
    classified = all_data[codename].get('classified')
    effects_df  = all_data[codename].get('effects')
    baselines_df = all_data[codename].get('baselines')
    if classified is None or baselines_df is None:
        continue

    # Exclude session days
    session_dates = set()
    if effects_df is not None and 'date' in effects_df.columns:
        session_dates = {pd.Timestamp(d).date() for d in effects_df['date'].dropna()}
    elif effects_df is not None and effects_df.index.dtype == 'datetime64[ns]':
        session_dates = {ts.date() for ts in effects_df.index}

    non_session = classified[~classified.index.to_series().apply(lambda ts: ts.date() in session_dates)]
    if 'activity_state' not in non_session.columns or 'stress' not in non_session.columns:
        print(f"{codename}: missing columns for recovery window plot")
        continue

    effort = non_session['activity_state'].map(STATE_ORDER).fillna(1)
    prev_effort = effort.shift(1)
    trans_mask = effort < prev_effort
    trans_times = non_session.index[trans_mask]

    # Collect recovery windows per from_state
    windows_by_state: dict = {}
    for ts in trans_times:
        prev_ts = ts - pd.Timedelta(minutes=1)
        if prev_ts not in non_session.index:
            continue
        from_state = non_session.at[prev_ts, 'activity_state']
        stay_start = ts - pd.Timedelta(minutes=3)
        prior = non_session.loc[stay_start:prev_ts, 'activity_state']
        if len(prior) < 3 or not (prior == from_state).all():
            continue
        window_end = ts + pd.Timedelta(minutes=90)
        window = non_session.loc[ts:window_end, 'stress'].dropna()
        if len(window) < 10:
            continue
        windows_by_state.setdefault(from_state, []).append(window)

    # Only plot states that also have a valid baseline
    stress_bl = baselines_df[baselines_df['signal'] == 'stress']
    valid_bl = stress_bl[stress_bl['tau_min'] < 490]
    states_to_plot = [s for s in valid_bl['from_state'].values if s in windows_by_state]
    if not states_to_plot:
        print(f"{codename}: geen herstelvensters gevonden voor weergave")
        continue

    n_cols = len(states_to_plot)
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 4), squeeze=False)
    emoji = PARTICIPANTS[codename]['emoji']
    fig.suptitle(
        f'{emoji} {codename} — herstelvensters op niet-sessiedagen (grijs) + gefitte curve (wit)',
        fontsize=11
    )

    for j, state in enumerate(states_to_plot):
        ax = axes[0][j]
        bl_row = valid_bl[valid_bl['from_state'] == state].iloc[0]
        asymptote = float(bl_row['asymptote'])
        tau_bl    = float(bl_row['tau_min'])
        n_obs     = int(bl_row['n_obs'])
        r2_bl     = float(bl_row['r_squared'])

        windows = windows_by_state[state][:MAX_WINDOWS]
        for win in windows:
            t_win = np.arange(len(win), dtype=float)
            ax.plot(t_win, win.values, color='#586475', linewidth=0.8, alpha=0.35)

        # Fitted baseline curve (uses asymptote + tau from baselines.csv)
        t_full = np.arange(0, 91, 1.0)
        if windows:
            start_vals = [float(w.iloc[0]) for w in windows]
            start_mean = float(np.mean(start_vals))
        else:
            start_mean = asymptote + 20
        y_bl = asymptote + (start_mean - asymptote) * np.exp(-t_full / tau_bl)
        ax.plot(t_full, y_bl, color='white', linewidth=2.0, linestyle='--', alpha=0.9,
                label=f'τ={tau_bl:.0f} min (n={n_obs}, r²={r2_bl:.2f})')

        # Asymptote — the "floor" the curve recovers toward
        ax.axhline(asymptote, color='#56B4E9', linewidth=1.0, linestyle=':', alpha=0.7,
                   label=f'Rustniveau (asymptoot = {asymptote:.0f})')

        ax.set_xlabel('Minuten na statusovergang')
        ax.set_ylabel('Stress')
        ax.set_title(f'Préstaat: {state}  ({len(windows)} vensters)', fontsize=10)
        ax.legend(fontsize=7, loc='upper right')

    plt.tight_layout()
    out_path = ANALYSIS_ROOT / f'recovery_windows_{codename}.png'
    plt.savefig(out_path, bbox_inches='tight', dpi=150)
    plt.show()
    print(f"Opgeslagen: {out_path}")

---

## Persoonlijke baseline per activatieniveau

Dit is het concept uit de onderzoeksplanning: *"Baseline per persoon uitwerken met verschillende activatieniveaus, zodat je kan zien welk activatieniveau voor de sessie bezig was en of muziek invloed heeft op normaal herstel."*

De pijplijn doet dit in drie stappen:

**Stap 1** — elke minuut een activatieniveau geven (`classified_minutes.csv`)

**Stap 2** — op niet-sessiedagen de herstelcurve per activatieniveau bepalen (`recovery_baselines.csv`)

**Stap 3** — voor elke sessie vergelijken: verwachte hersteltijd vs. werkelijke hersteltijd

| Kolom | Betekenis |
|-------|-----------|
| `pre_state` | Activatieniveau vóór de sessie |
| `tau_expected` | Verwachte hersteltijd uit de baseline |
| `tau_actual` | Werkelijke hersteltijd tijdens de sessie |
| `advantage` | tau_expected min tau_actual (positief = muziek hielp) |

**De grafiek hieronder** toont per deelnemer en per activatieniveau de gestippelde witte baseline tegenover de gekleurde sessielijnen. Hoe verder een gekleurde lijn onder de baseline zit, hoe groter het voordeel.

### Concreet voorbeeld: van baseline naar voordeel

Voordat je de grafiek bekijkt, een uitgewerkt voorbeeld met echte getallen:

```
bosbes | Energy-sessie 2025-01-29

  Rustniveau (asymptoot):          ~30 stress
  Préstaat:                        Slaap
  Normale herstellingssnelheid:    τ = 97.8 min  ← hoelang stress normaal naar ~30 daalt
  Herstellingssnelheid met muziek: τ =  3.4 min  ← hoelang het tijdens deze sessie duurde
  ─────────────────────────────────────────────
  Voordeel:                       +94.4 min  ← bosbes bereikte het rustniveau ~95 min sneller
  r²:                               0.82     ← goede fit, resultaat is betrouwbaar
```

**Wat de grafiek hieronder laat zien:** de witte stippellijn is de normale herstellingscurve (τ = 97.8 min). De gekleurde lijnen zijn de werkelijke sessies. Hoe lager een gekleurde lijn zit ten opzichte van de stippellijn, hoe groter het voordeel.

In [ ]:
**Wat je ziet:**

Een gemiddeld voordeel van **+48 minuten** klinkt groot. Maar de **standaardafwijking van 58 minuten** is groter dan het gemiddelde: de resultaten zijn sterk wisselend. Sommige sessies tonen een groot voordeel, andere een negatief resultaat.

Energy scoort hoger dan Calm voor de huidige deelnemers, maar met 6 tot 7 sessies per type is dat nog geen statistisch bewijs.

> **Kanttekening**: Sessies met **r² bijna nul** zitten nog in het gemiddelde hieronder. Het `reliable`-scorebord bovenaan toont het gefilterd gemiddelde.

---

### Methodologische beperking — Energy vs Calm zijn niet goed te vergelijken met dit meetgetal

De code behandelt alle drie afspeellijsttypen **identiek**: elk type gebruikt dezelfde tau-berekening op het stresssignaal. Dat is een probleem:

- Een **Calm**-playlist is bedoeld om arousal te verlagen → snellere stressdaling is logisch → tau daalt → positief voordeel. ✓ Consistent met het meetgetal.
- Een **Energy**-playlist is bedoeld om alertheid te *verhogen* → je zou eigenlijk verwachten dat het stresssignaal *hoog blijft of stijgt*. Toch toont Energy de grootste voordelen in de huidige data.

**Mogelijke verklaringen:**
1. Het Garmin-stresssignaal is HRV-gebaseerd (hartslagvariabiliteit), niet arousaal. Elke positieve muzikale betrokkenheid — ook energieke — kan HRV stabiliseren, ongeacht richting.
2. De meeste Energy-sessies van bosbes vallen in de vroege ochtend (pre_state = Slaap, hoge tau_verwacht). Het dagtijdstip voorspelt het voordeel dan beter dan het afspeellijsttype.

**Conclusie:** het huidige meetgetal kan *niet* onderscheiden of een Energy-playlist werkte *omdat hij energetisch is*, of gewoon *omdat je 's ochtends sowieso snel herstelt*. Om dat onderscheid te maken, zou je een apart arousalmeetgetal nodig hebben — bv. een stijging in hartslag of zelfgerapporteerde energie na de sessie.

if all_effects.empty:
    print("No cross-participant effects found. Run pipeline.py first.")
else:
    valid_eff = all_effects.dropna(subset=['advantage'])
    print(f"Voordeel-scores aanwezig: {len(valid_eff)} / {len(all_effects)}")
    print(f"  Gemiddeld (alle):       {valid_eff['advantage'].mean():+.2f} min  ← inclusief ruis")
    if not reliable.empty:
        print(f"  Gemiddeld (reliable):   {reliable['advantage'].mean():+.2f} min  ← gebruik dit")
        print(f"  SD (reliable):          {reliable['advantage'].std():.2f} min")
    print()
    if not reliable.empty:
        print("Betrouwbare sessies per afspeellijst (r² > 0.05 + pre_stress ≥ asymptoot):")
        print(reliable.groupby('playlist')['advantage'].agg(['mean', 'std', 'count']).round(2))
    else:
        print(valid_eff.groupby('playlist')['advantage'].agg(['mean', 'std', 'count']).round(2))

In [ ]:
if all_effects.empty:
    print("No cross-participant effects found. Run pipeline.py first.")
else:
    valid_eff = all_effects.dropna(subset=['advantage'])
    print(f"Valid advantage scores: {len(valid_eff)} / {len(all_effects)}")
    print(f"Mean advantage: {valid_eff['advantage'].mean():+.2f} min")
    print(f"Std:            {valid_eff['advantage'].std():.2f} min")
    print()
    print(valid_eff.groupby('playlist')['advantage'].agg(['mean', 'std', 'count']).round(2))

**Wat je ziet:**

Een gemiddeld voordeel van **+48 minuten** klinkt groot. Maar de **standaardafwijking van 58 minuten** is groter dan het gemiddelde: de resultaten zijn sterk wisselend. Sommige sessies tonen een groot voordeel, andere een negatief resultaat.

Energy scoort hoger dan Calm voor de huidige deelnemers, maar met 6 tot 7 sessies per type is dat nog geen statistisch bewijs.

> **Kanttekening**: Sessies met **r² bijna nul** zitten nog in het gemiddelde. Als die worden uitgefilterd, verandert het gemiddelde sterk. Dit filter staat als prioritaire volgende stap.

In [ ]:
if not all_effects.empty and 'advantage' in all_effects.columns:
    valid_eff = all_effects.dropna(subset=['advantage'])

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('Recovery Advantage: Music Sessions vs Expected Recovery', fontsize=12)

    # 1. Distribution of advantage scores
    ax = axes[0]
    ax.axvline(0, color='#D55E00', linewidth=1.5, linestyle='--', label='No effect')
    for codename, group in valid_eff.groupby('participant'):
        color = PARTICIPANTS.get(codename, {}).get('color', '#888')
        ax.hist(group['advantage'], bins=10, alpha=0.6, color=color, label=codename)
    ax.set_xlabel('Recovery Advantage (min)\nPositive = faster recovery with music')
    ax.set_ylabel('Sessions')
    ax.set_title('Distribution')
    ax.legend(fontsize=7)

    # 2. Advantage by playlist type
    ax = axes[1]
    playlists = valid_eff['playlist'].dropna().unique()
    playlist_colors = {'Calm': '#56B4E9', 'Energy': '#D55E00', 'Neutral': '#F0E442'}
    positions = np.arange(len(playlists))
    for k, pl in enumerate(sorted(playlists)):
        sub = valid_eff[valid_eff['playlist'] == pl]['advantage']
        ax.boxplot(sub, positions=[k], widths=0.5,
                   patch_artist=True,
                   boxprops={'facecolor': playlist_colors.get(pl, '#888'), 'alpha': 0.7},
                   medianprops={'color': 'white', 'linewidth': 2},
                   whiskerprops={'color': '#586475'},
                   capprops={'color': '#586475'},
                   flierprops={'marker': 'o', 'color': '#586475', 'markersize': 4})
    ax.axhline(0, color='#D55E00', linewidth=1, linestyle='--')
    ax.set_xticks(range(len(sorted(playlists))))
    ax.set_xticklabels(sorted(playlists))
    ax.set_ylabel('Recovery Advantage (min)')
    ax.set_title('By Playlist Type')

    # 3. Advantage by pre-session state
    ax = axes[2]
    states_present = valid_eff['pre_state'].dropna().unique()
    for k, state in enumerate(sorted(states_present)):
        sub = valid_eff[valid_eff['pre_state'] == state]['advantage']
        if len(sub) < 2:
            continue
        ax.boxplot(sub, positions=[k], widths=0.5,
                   patch_artist=True,
                   boxprops={'facecolor': STATE_COLORS.get(state, '#888'), 'alpha': 0.7},
                   medianprops={'color': 'white', 'linewidth': 2},
                   whiskerprops={'color': '#586475'},
                   capprops={'color': '#586475'},
                   flierprops={'marker': 'o', 'color': '#586475', 'markersize': 4})
    ax.axhline(0, color='#D55E00', linewidth=1, linestyle='--')
    ax.set_xticks(range(len(sorted(states_present))))
    ax.set_xticklabels(sorted(states_present), rotation=30, ha='right')
    ax.set_ylabel('Recovery Advantage (min)')
    ax.set_title('By Pre-session Activity State')

    plt.tight_layout()
    plt.savefig(ANALYSIS_ROOT / 'recovery_advantage.png', bbox_inches='tight', dpi=150)
    plt.show()

**De grafieken lezen:**

**Links: Verdeling**
Elke balk = aantal sessies in dat voordeel-bereik. Rechts van de stippellijn = sneller herstel, links = trager. Bosbes = hemelblauw, kokosnoot = oranje.

**Midden: Per type afspeellijst**
**Boxplots**: de witte lijn is de mediaan, de box beslaat de middelste 50%, de snorharen zijn min/max. Als de box boven de stippellijn ligt, helpt dat type afspeellijst gemiddeld.

**Rechts: Per préstaat**
Zelfde boxplots maar per activatieniveau vóór de sessie. Een groot verschil hier betekent dat muziek beter werkt vanuit bepaalde toestanden.

> Met 6 tot 9 sessies per deelnemer zijn deze boxplots illustratief, niet statistisch conclusief.

import json

stats_path = ANALYSIS_ROOT / 'cross_participant_stats.json'
if stats_path.exists():
    with open(stats_path) as f:
        stat_results = json.load(f)

    print('=== One-sample t-test (advantage ≠ 0) — ALLE sessies ===')
    if 'ttest' in stat_results:
        t = stat_results['ttest']
        print(f"  n = {t['n']}  (inclusief r²≤0.05 en pre_stress<asymptoot)")
        print(f"  Mean advantage = {t['mean_advantage_min']:+.2f} min (SD={t['std_advantage']:.2f})")
        print(f"  t = {t['statistic']:.3f}, p = {t['p_value']:.4f}")
        print(f"  → {t['interpretation']}")
        print()

    print('=== ANOVA by playlist type ==='  )
    if 'anova_playlist' in stat_results:
        a = stat_results['anova_playlist']
        print(f"  F({a['n_groups']-1}, N) = {a['f_statistic']:.3f}, p = {a['p_value']:.4f}")
        print()

    if 'tukey' in stat_results:
        print('=== Tukey HSD post-hoc ===')
        for pair, res in stat_results['tukey'].items():
            print(f"  {pair}: p = {res['p_value']:.4f}")
        print()

    print('=== Mixed-effects model ===')
    if 'mixed_effects' in stat_results:
        me = stat_results['mixed_effects']
        if 'error' in me:
            print(f"  Not available: {me['error']}")
        else:
            print(f"  Converged: {me.get('converged')}, AIC: {me.get('aic')}")
            print("  Coefficients:")
            for k, v in me.get('coefficients', {}).items():
                print(f"    {k:<40} coef={v['coef']:+.3f}  p={v['p_value']:.4f}")
else:
    print('No stats file found. Run pipeline.py first.')
    if not all_effects.empty:
        print("Running stats on loaded effects...")
        from session_effect import run_statistics
        stat_results = run_statistics(all_effects)
        if 'ttest' in stat_results:
            t = stat_results['ttest']
            print(f"  → {t['interpretation']} (p={t['p_value']:.4f})")

# ── T-toets op betrouwbare sessies (r² > 0.05 + pre_stress ≥ asymptoot) ──────
if not reliable.empty and len(reliable) > 1:
    t_stat, p_val = stats.ttest_1samp(reliable['advantage'], 0)
    sig = "statistisch significant" if p_val < 0.05 else "niet significant"
    print()
    print('=== T-toets op BETROUWBARE sessies (gebruik dit) ===')
    print(f"  n = {len(reliable)}  (na r²- en pre_stress-filter)")
    print(f"  Gemiddeld voordeel = {reliable['advantage'].mean():+.2f} min (SD={reliable['advantage'].std():.2f})")
    print(f"  t = {t_stat:.3f}, p = {p_val:.4f}")
    print(f"  → {sig}")

In [ ]:
import json

stats_path = ANALYSIS_ROOT / 'cross_participant_stats.json'
if stats_path.exists():
    with open(stats_path) as f:
        stat_results = json.load(f)

    print('=== One-sample t-test (advantage ≠ 0) ===')
    if 'ttest' in stat_results:
        t = stat_results['ttest']
        print(f"  n = {t['n']}")
        print(f"  Mean advantage = {t['mean_advantage_min']:+.2f} min (SD={t['std_advantage']:.2f})")
        print(f"  t = {t['statistic']:.3f}, p = {t['p_value']:.4f}")
        print(f"  → {t['interpretation']}")
        print()

    print('=== ANOVA by playlist type ===')
    if 'anova_playlist' in stat_results:
        a = stat_results['anova_playlist']
        print(f"  F({a['n_groups']-1}, N) = {a['f_statistic']:.3f}, p = {a['p_value']:.4f}")
        print()

    if 'tukey' in stat_results:
        print('=== Tukey HSD post-hoc ===')
        for pair, res in stat_results['tukey'].items():
            print(f"  {pair}: p = {res['p_value']:.4f}")
        print()

    print('=== Mixed-effects model ===')
    if 'mixed_effects' in stat_results:
        me = stat_results['mixed_effects']
        if 'error' in me:
            print(f"  Not available: {me['error']}")
        else:
            print(f"  Converged: {me.get('converged')}, AIC: {me.get('aic')}")
            print("  Coefficients:")
            for k, v in me.get('coefficients', {}).items():
                print(f"    {k:<40} coef={v['coef']:+.3f}  p={v['p_value']:.4f}")
else:
    print('No stats file found. Run pipeline.py first.')
    if not all_effects.empty:
        print("Running stats on loaded effects...")
        from session_effect import run_statistics
        stat_results = run_statistics(all_effects)
        if 'ttest' in stat_results:
            t = stat_results['ttest']
            print(f"  → {t['interpretation']} (p={t['p_value']:.4f})")

**Wat de uitvoer zegt (p = 0.0112):**

De **t-toets** geeft p = 0.011, technisch onder de drempel van 0.05. Conclusie: *"Muziekluistersessies tonen significant snellere recuperatie."*

**Maar voorzichtig:**
- n = 13 sessies is erg klein
- Alle sessies zijn van slechts 2 deelnemers (niet echt onafhankelijk)
- De grote voordelen van bosbes (+121, +121, +108, +101) drijven het gemiddelde; kokosnoot alleen zou niet significant zijn
- Sessies met **r² bijna nul** zijn nog niet gefilterd

**Eerlijk**: het signaal is veelbelovend genoeg om verder te gaan. Het is nog niet sterk genoeg voor een publiceerbare conclusie.

if not all_effects.empty and 'tau_expected' in all_effects.columns:
    valid = all_effects.dropna(subset=['tau_expected', 'tau_actual']).copy()
    valid = valid.sort_values('tau_expected', ascending=False).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(14, 5))
    x = np.arange(len(valid))
    ax.bar(x, valid['tau_expected'], color='#232b3a', label='τ verwacht (baseline)', zorder=2)

    playlist_colors = {'Calm': '#56B4E9', 'Energy': '#D55E00', 'Neutral': '#F0E442'}
    for i, row in valid.iterrows():
        color = playlist_colors.get(row.get('playlist', ''), '#888')
        r2 = float(row.get('r2_actual', 0))
        alpha = 0.85 if r2 > 0.05 else 0.25  # fade unreliable sessions
        ax.bar(i, row['tau_actual'], color=color, alpha=alpha, zorder=3)

    # x-axis: participant abbreviation + playlist type + reliability dot
    labels = []
    for _, row in valid.iterrows():
        p  = str(row.get('participant', ''))[:3]
        pl = str(row.get('playlist',    ''))[:3]
        r2 = float(row.get('r2_actual', 0))
        q  = '●' if r2 > 0.05 else '○'   # filled circle = reliable
        labels.append(f"{q}{p}\n{pl}")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=7)

    ax.set_xlabel('Sessie (gesorteerd op verwachte hersteltijd) · ●=betrouwbaar  ○=onbetrouwbaar (r²≤0.05)')
    ax.set_ylabel('Herstel-τ (min)')
    ax.set_title('Verwachte vs werkelijke tijdconstante per sessie')

    from matplotlib.patches import Patch
    handles = [Patch(facecolor=c, label=pl) for pl, c in playlist_colors.items()]
    handles += [
        Patch(facecolor='#aaa', alpha=0.85, label='betrouwbaar (r²>0.05)'),
        Patch(facecolor='#aaa', alpha=0.25, label='onbetrouwbaar (r²≤0.05)'),
    ]
    ax.legend(handles=handles, fontsize=8, loc='upper right')
    plt.tight_layout()
    plt.savefig(ANALYSIS_ROOT / 'tau_waterfall.png', bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
if not all_effects.empty and 'tau_expected' in all_effects.columns:
    valid = all_effects.dropna(subset=['tau_expected', 'tau_actual']).copy()
    valid = valid.sort_values('tau_expected', ascending=False).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(14, 5))
    x = np.arange(len(valid))
    ax.bar(x, valid['tau_expected'], color='#232b3a', label='τ expected (baseline)', zorder=2)
    ax.bar(x, valid['tau_actual'], color='#56B4E9', alpha=0.7, label='τ actual (during music)', zorder=3)

    # Colour by playlist (Okabe-Ito)
    playlist_colors = {'Calm': '#56B4E9', 'Energy': '#D55E00', 'Neutral': '#F0E442'}
    for i, row in valid.iterrows():
        color = playlist_colors.get(row.get('playlist', ''), '#888')
        ax.bar(i, row['tau_actual'], color=color, alpha=0.8, zorder=3)

    ax.set_xlabel('Session (sorted by expected recovery time)')
    ax.set_ylabel('Recovery τ (min)')
    ax.set_title('Expected vs Actual Recovery Time Constant per Session')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(ANALYSIS_ROOT / 'tau_waterfall.png', bbox_inches='tight', dpi=150)
    plt.show()

**Wat je ziet:**

- **Links** (hoge tau-verwacht): ochtendse Energy-sessies van bosbes. Baseline ~98 min, werkelijk ~2 à 3 min → groot voordeel van +95 min.
- **Rechts** (lage tau-verwacht): sessies van kokosnoot en de Rust-sessie van bosbes. Voordelen kleiner.
- **Sessies waarbij de gekleurde staaf hoger is**: negatieve voordelen. Als **r² ≈ 0**, is de gekleurde staaf ruis.

**Bosbes, 30 jan, Calm (tau-werkelijk > tau-verwacht):** bosbes was al kalmer dan de asymptoot vóór de sessie. De exponentiële formule werkt dan niet meer en geeft een artefact.

> **Kanttekening**: sessies waarbij **pre_stress < asymptoot** zouden automatisch uitgefilterd moeten worden in `session_effect.py`. Dit is een eenvoudige prioritaire fix.

---

## Stemming-delta versus herstelvoordeel

Voelen deelnemers zich beter na sessies waarbij ze ook fysiologisch sneller herstelden?

- **x-as**: **tau-voordeel** (positief = sneller herstel)
- **y-as**: **stemming-delta** = stemming na min stemming voor (positief = beter gevoel)
- Stippellijnen bij 0,0 verdelen in vier kwadranten:
  - **Rechtsboven**: sneller hersteld én beter gevoel (ideaal)
  - **Linksboven**: beter gevoel, geen fysiologisch voordeel
  - **Rechtsonder**: sneller hersteld, maar slechter gevoel
  - **Linksonder**: geen voordeel op beide assen
- **r** en **p** in de titel = **Pearson-correlatie** en bijbehorende p-waarde

> Verdieping: Russell (1980) *A circumplex model of affect*, Journal of Personality and Social Psychology.

In [ ]:
if not all_effects.empty and 'mood_delta' in all_effects.columns:
    scatter = all_effects.dropna(subset=['advantage', 'mood_delta'])

    if len(scatter) >= 5:
        fig, ax = plt.subplots(figsize=(7, 5))
        for codename, group in scatter.groupby('participant'):
            color = PARTICIPANTS.get(codename, {}).get('color', '#888')
            ax.scatter(group['advantage'], group['mood_delta'], color=color,
                       alpha=0.8, s=60, label=codename)

        # Regression line
        r, p = stats.pearsonr(scatter['advantage'], scatter['mood_delta'])
        m, b = np.polyfit(scatter['advantage'], scatter['mood_delta'], 1)
        x_line = np.linspace(scatter['advantage'].min(), scatter['advantage'].max(), 100)
        ax.plot(x_line, m * x_line + b, color='white', linewidth=1.5, alpha=0.6)

        ax.axvline(0, color='#586475', linewidth=0.8, linestyle='--')
        ax.axhline(0, color='#586475', linewidth=0.8, linestyle='--')
        ax.set_xlabel('Recovery Advantage (min)\nPositive = faster recovery with music')
        ax.set_ylabel('Mood Delta (after − before)')
        ax.set_title(f'Physiological Recovery vs Mood Change\nr={r:.3f}, p={p:.4f} (n={len(scatter)})')
        ax.legend(fontsize=7)
        plt.tight_layout()
        plt.savefig(ANALYSIS_ROOT / 'recovery_vs_mood.png', bbox_inches='tight', dpi=150)
        plt.show()
    else:
        print('Not enough data points with both advantage and mood_delta.')

**Wat je ziet:**

Met ~13 punten is elke correlatie hier onzeker. Een paar sessies om te checken:

- **bosbes, Energy-sessies (+121 voordeel, +1 à +2 stemming)**: rechtsboven, consistent positief.
- **bosbes, 30 jan Calm (−70 voordeel, −1 stemming)**: linksonder. Het artefact van de te-lage pre_stress trekt ook de stemming-delta mee naar beneden.
- **bosbes, 10 mrt Energy (+101 voordeel, −2 stemming)**: rechtsonder. Fysiologisch hersteld, maar toch slechter gevoel achteraf.

Als de regressielijn positief helt en r > 0, is er een richtinggevende afstemming. Zelfs een zwakke correlatie bij dit n is het vermelden waard.

> **Kanttekening**: stemmingsscores lopen van ~4 tot 9 (niet 0 tot 10). De **stemming-delta** is daardoor samengedrukt en niet direct vergelijkbaar met publicaties die een gestandaardiseerde schaal gebruiken.

---

## Samenvatting en discussie

### Kernresultaten

| Meting | Waarde |
|--------|--------|
| Deelnemers met effecten | bosbes (7), kokosnoot (9), limoen (1) |
| Geldige **voordeel-scores** | 13 van 17 sessies |
| Betrouwbare fits (r² > 0.05) | 8 |
| Gemiddeld voordeel | **+48.5 min** (SD = 58 min) |
| Gemiddeld voordeel (alleen r² > 0.05) | **~+54 min** |
| t-toets p-waarde | **p = 0.011** (n = 13, technisch significant) |
| Voordeel bosbes | ~+75 min (6 sessies, 5 betrouwbaar) |
| Voordeel kokosnoot | ~+25 min (7 sessies, 3 betrouwbaar) |

### Wat de data ondersteunt

- **Muziek versnelt stressherstel bij bosbes** als de deelnemer gestresseerd was vóór de sessie. Effect: +70 tot +121 min over 5 betrouwbare sessies.
- **pre_stress_mean is de sterkste voorspeller**: weinig stress voor de sessie = weinig voordeel. Dit klopt met het **ISO-principe**: muziekregulatie werkt door te overbruggen van verhoogde arousal naar een doeltoestand. Als je al rustig bent, is er niets te overbruggen.
- **kokosnoot toont een positief gemiddeld voordeel (+25 min)**, maar slechts 3 van 7 fits zijn betrouwbaar (r² > 0.05). De stemmingsdata (16 sessies) biedt aanvullende informatie voor deze deelnemer.

### Beperkingen

1. **r² ≈ 0** voor verscheidene sessies: het exponentiële model past niet altijd goed. LOWESS of een tweeledig model zou beter zijn.
2. **Kleine steekproef**: 3 deelnemers, 13 geldige sessies. Resultaten zijn voorlopige signalen.
3. **Geen automatisch pre_stress-filter**: sessies waarbij pre_stress < asymptoot geven een modelartefact en vertekenen het gemiddelde neerwaarts.
4. **Individuele session_effects.csv-bestanden** zijn van een oudere pijplijnrun en wijken af van `cross_participant_effects.csv`. Herrun de pijplijn om ze te synchroniseren.

### Prioritaire volgende stappen

| Prioriteit | Actie |
|-----------|-------|
| Hoog | Filter **r2_actual < 0.05** automatisch uit alle voordeel-berekeningen |
| Hoog | Filter sessies waarbij **pre_stress < asymptoot** |
| Hoog | Voer pijplijn uit op **peer** zodra data beschikbaar is |
| Hoog | Voeg **hartfrequentie** toe als terugval-signaal voor schaarse stressdata |
| Middel | Onderzoek niet-exponentieel herstelpatroon bosbes (LOWESS?) |
| Lager | Train activiteitsclassificator op FIT-intensiteitslabels |

> Verdieping: Thoma et al. (2013) *The effect of music on the human stress response*, PLOS ONE.